# Threshold Evaluation

This notebook evaluates LPT-R binarization applied to species range predictions from an iNat geo model. We generate apply F1 and IOU metrics for predicted maps vs SNT expert ranges.

Model: tf_geoprior_2_20_r6_elevation.h5  
Elevation raster: elev_scaled.npy  
Taxonomy: vision-export-20250119070012-aka-2.20  
Range source: SNT

In [56]:
import os
from pathlib import Path
import sys

import h3
import h3pandas
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydeck as pdk
from shapely.geometry import shape
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import average_precision_score
import tensorflow as tf
from tqdm.auto import tqdm

In [2]:
sys.path.append('..')
from lib.models.tf_gp_model import TFGeoPriorModel
from lib.models.encoders import CoordEncoder

In [406]:
GEOMODEL_PATH = "/home/alex/export_eval_models/model_exports/tf_geoprior_2_20_r6_elevation.h5"
ELEVATION_RASTER_PATH = "/home/alex/sinr/data/env/elev_scaled.npy"
INPUT_ENCODING = "sinusoidal"
TAX_2_20_FILE = "/data-ssd/alex/datasets/vision-export-20250119070012-aka-2.20/taxonomy.csv"
SPATIAL_DATA_2_20_FILE = "/data-ssd/alex/datasets/vision-export-20250119070012-aka-2.20/spatial_data.parquet"
SNT_PATH = "/home/alex/sinr/data/eval/snt/snt_res_5.npy"
GRID_THRESHOLDS = "/home/alex/inatVisionApi/geo_model_eval_output/v2_20_grid/thresholds.csv"

## 1. Load Model and Setup

In [407]:
model = TFGeoPriorModel(GEOMODEL_PATH)
raster = np.load(ELEVATION_RASTER_PATH).astype(np.float32)
encoder = CoordEncoder(INPUT_ENCODING, raster=raster)
tax_2_20 = pd.read_csv(TAX_2_20_FILE)
inat_standard_thresholds = pd.read_csv(GRID_THRESHOLDS)

In [408]:
inat_standard_thresholds

,Unnamed: 0,taxon_id,thres,area
0,0,152875,6.111034e-05,1.970547e+07
1,1,48272,1.111355e-04,1.720223e+07
2,2,81614,4.317025e-05,6.385557e+06
3,3,712964,3.500609e-05,1.428651e+07
4,4,472689,4.106176e-03,1.200279e+06
...,...,...,...,...
100343,100343,783080,1.124506e-05,5.139249e+07
100344,100344,1145193,7.021856e-07,1.963979e+08
100345,100345,1538530,2.065940e-04,5.385324e+06
100346,100346,1538531,9.564230e-05,5.758863e+06


## 2. Load SNT data for eval

In [179]:
SNT_data = np.load(SNT_PATH, allow_pickle=True)
SNT_data = SNT_data.item()

In [184]:
SNT_taxon_ids = SNT_data['taxa']
SNT_loc_indices_per_species = SNT_data['loc_indices_per_species']
SNT_labels_per_species = SNT_data['labels_per_species']
SNT_obs_locs = np.array(SNT_data['obs_locs'], dtype=np.float32)

In [185]:
SNT_taxon_ids[:3]

array([  7, 162, 243], dtype=int32)

In [186]:
SNT_labels_per_species[0][:5]

[0, 0, 0, 0, 0]

In [187]:
SNT_obs_locs[0][:5]

array([11.927741, 62.76889 ], dtype=float32)

## 3. make predictions at SNT locations

In [188]:
encoded_locs = encoder.encode(SNT_obs_locs)
geo_model_features = model.get_loc_emb(encoded_locs)

In [190]:
target_taxon_id = SNT_taxon_ids[0]
target_taxon_id

7

In [191]:
target_class_id = int(
    tax_2_20[
        tax_2_20.taxon_id==target_taxon_id
    ].iloc[0].leaf_class_id
)
target_class_id

48489

In [263]:
preds = model.eval_one_class_from_feats(
    geo_model_features,
    class_of_interest=target_class_id
)[0]
preds.shape

(554882,)

In [264]:
SNT_cur_loc_indices = np.array(SNT_loc_indices_per_species[0])
SNT_y_test = np.array(SNT_labels_per_species[0])

In [266]:
SNT_y_test.shape, preds.shape, y_thresh.shape

((174746,), (554882,), ())

In [267]:
preds = preds[SNT_cur_loc_indices]
preds.shape

(174746,)

In [302]:
y_thresh = preds>0.01
y_thresh[0]

False

In [303]:
SNT_y_test[0]

0

In [304]:
f1_score(SNT_y_test, y_thresh, average="binary")

0.5358636217433258

In [251]:
dfh3 = pd.DataFrame(
    SNT_obs_locs,
    columns=["lng", "lat"],
).h3.geo_to_h3(resolution=4)
dfh3.sample(5)

,lng,lat
h3_04,,
84136cdffffffff,-99.955208,65.606949
8497853ffffffff,36.334785,-14.222179
849613dffffffff,31.569029,-10.427502
84bcce5ffffffff,26.554228,-25.911154
8420c1bffffffff,82.616234,41.878239


In [33]:
dfh3["preds"] = preds[0]

In [34]:
dfh3

,lng,lat,preds
h3_04,,,
8408101ffffffff,11.927741,62.768890,2.021607e-09
8408101ffffffff,12.061964,62.876320,1.516412e-09
8408101ffffffff,12.089678,62.680466,2.039185e-09
8408101ffffffff,12.224094,62.787971,1.804204e-09
8408101ffffffff,11.631933,62.749199,2.001996e-09
...,...,...,...
845b43bffffffff,167.756897,8.762376,3.216453e-05
845b4e7ffffffff,166.840271,9.257710,1.278451e-05
845b4e7ffffffff,166.686066,9.260979,1.257822e-08


In [116]:
dfh3['binary_0_1'] = (dfh3['preds'] >= 0.05).astype(int)
dfh3.sample(5)

,lng,lat,preds,binary_0_1
h3_04,,,,
848a8c7ffffffff,-57.818329,-2.678905,1.216646e-02,0
8426629ffffffff,-88.980995,40.772808,2.862272e-03,0
840a0b5ffffffff,96.468979,66.289909,2.876232e-10,0
8402e47ffffffff,-103.375435,75.157181,3.456581e-08,0
841b92bffffffff,-61.173496,50.086548,1.438464e-07,0


In [117]:
(
    dfh3.shape, 
    len(SNT_loc_indices_per_species[0]), 
    SNT_obs_locs.shape,
    
)

((554882, 4), 174746, (554882, 2))

In [118]:
SNT_obs_locs.shape

(554882, 2)

In [119]:
split_rng = np.random.default_rng()
cur_loc_indices = np.array(SNT_loc_indices_per_species[0])
cur_labels = np.array(SNT_labels_per_species[0])
SNT_val_frac = 0.5
num_val = np.floor(len(cur_labels) * 0.5).astype(int)
idx_rand = split_rng.permutation(len(cur_labels))

idx_sel = idx_rand[num_val:]
cur_loc_indices = cur_loc_indices[idx_sel]
cur_labels = cur_labels[idx_sel]

pred = dfh3.iloc[cur_loc_indices]

f1_score(
    (cur_labels > 0).astype(np.int32), pred["binary_0_1"]
)

0.19891196076928971

In [414]:
f1_scores = {}

for i in tqdm(range(100)):
    target_taxon_id = SNT_taxon_ids[i]
    if target_taxon_id in tax_2_20.taxon_id.values:
        target_class_id = int(
            tax_2_20[
                tax_2_20.taxon_id==target_taxon_id
            ].iloc[0].leaf_class_id
        )
        preds = model.eval_one_class_from_feats(
            geo_model_features,
            class_of_interest=target_class_id
        )
        dfh3 = pd.DataFrame(
            SNT_obs_locs,
            columns=["lng", "lat"],
        )
        dfh3["preds"] = preds[0]

        thresholds = {
            "binary_0_0001": 0.0001,
            "binary_0_001": 0.001,
            "binary_0_01": 0.01,
            "binary_0_1": 0.1,
            "binary_0_5": 0.5,
        }

        inat_thresh = inat_standard_thresholds[
            inat_standard_thresholds.taxon_id==target_taxon_id
        ].iloc[0].thres
        thresholds["inat_standard"] = inat_thresh


        split_rng = np.random.default_rng()
        cur_loc_indices = np.array(SNT_loc_indices_per_species[0])
        cur_labels = np.array(SNT_labels_per_species[0])
        SNT_val_frac = 0.5
        num_val = np.floor(len(cur_labels) * 0.5).astype(int)
        idx_rand = split_rng.permutation(len(cur_labels))
        
        idx_sel = idx_rand[num_val:]
        cur_loc_indices = cur_loc_indices[idx_sel]
        cur_labels = cur_labels[idx_sel]
        
        pred = dfh3.iloc[cur_loc_indices]

        f1_row = {}
        
        for name, threshold in thresholds.items():
            thresholded_preds = (pred['preds'] >= threshold).astype(int)
            
            f1 = f1_score(
                (cur_labels > 0).astype(np.int32), thresholded_preds
            )

            f1_row[name] = f1
        f1_scores[target_taxon_id] = f1_row
    else:
        print(f"{target_taxon_id} not found")


  0%|          | 0/100 [00:00<?, ?it/s]

5009 not found
5108 not found
5112 not found


In [415]:
pd.DataFrame(f1_scores).T.mean()

binary_0_0001    0.156316
binary_0_001     0.125069
binary_0_01      0.088739
binary_0_1       0.014017
binary_0_5       0.000425
inat_standard    0.124811
dtype: float64

In [416]:
pd.DataFrame(f1_scores).T

,binary_0_0001,binary_0_001,binary_0_01,binary_0_1,binary_0_5,inat_standard
7,0.389465,0.430263,0.540151,0.053377,0.000000,0.465781
162,0.103794,0.065363,0.031577,0.001433,0.000000,0.066219
243,0.024697,0.015915,0.007313,0.000000,0.000000,0.017023
473,0.121093,0.070980,0.051169,0.025139,0.000000,0.055030
519,0.388574,0.448147,0.474137,0.021086,0.000000,0.476018
...,...,...,...,...,...,...
5206,0.047098,0.046723,0.022246,0.026144,0.023264,0.026869
5212,0.091042,0.045646,0.038925,0.011542,0.000533,0.044171
5227,0.278692,0.136012,0.052548,0.006116,0.000000,0.127668
5268,0.080246,0.001342,0.000000,0.000000,0.000000,0.000000


In [417]:
pd.DataFrame(f1_scores).T.describe()

,binary_0_0001,binary_0_001,binary_0_01,binary_0_1,binary_0_5,inat_standard
count,97.000000,97.000000,97.000000,97.000000,97.000000,97.000000
mean,0.156316,0.125069,0.088739,0.014017,0.000425,0.124811
std,0.137290,0.142579,0.133919,0.020439,0.002385,0.145397
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.034009,0.013127,0.002666,0.000178,0.000000,0.012584
50%,0.119244,0.062362,0.038012,0.003874,0.000000,0.052726
75%,0.273826,0.217494,0.087461,0.021086,0.000000,0.221260
max,0.440657,0.463069,0.540151,0.098000,0.023264,0.476018


In [419]:
ap_scores = {}

for i in tqdm(range(100)):
    target_taxon_id = SNT_taxon_ids[i]
    if target_taxon_id in tax_2_20.taxon_id.values:
        target_class_id = int(
            tax_2_20[
                tax_2_20.taxon_id==target_taxon_id
            ].iloc[0].leaf_class_id
        )
        preds = model.eval_one_class_from_feats(
            geo_model_features,
            class_of_interest=target_class_id
        )
        dfh3 = pd.DataFrame(
            SNT_obs_locs,
            columns=["lng", "lat"],
        )
        dfh3["preds"] = preds[0]

        split_rng = np.random.default_rng()
        cur_loc_indices = np.array(SNT_loc_indices_per_species[0])
        cur_labels = np.array(SNT_labels_per_species[0])
        SNT_val_frac = 0.5
        num_val = np.floor(len(cur_labels) * 0.5).astype(int)
        idx_rand = split_rng.permutation(len(cur_labels))
        
        idx_sel = idx_rand[num_val:]
        cur_loc_indices = cur_loc_indices[idx_sel]
        cur_labels = cur_labels[idx_sel]
        
        pred = dfh3.iloc[cur_loc_indices]

        ap_score = average_precision_score(
            (cur_labels > 0).astype(np.int32),
            pred["preds"],
        )
        
        ap_scores[target_taxon_id] = ap_score
    else:
        print(f"{target_taxon_id} not found")
    

  0%|          | 0/100 [00:00<?, ?it/s]

5009 not found
5108 not found
5112 not found


In [429]:
np.array(list(ap_scores.values())).mean()

0.1341654730915568

## this isn't working because i need to load the SNT class to taxa i think?

In [458]:
SNT_taxon_ids[:10]

array([  7, 162, 243, 473, 519, 542, 804, 831, 890, 906], dtype=int32)